# pgvector II: HNSW indexes and filtered search

**Does pgvector create an index automatically? No.** Nothing indexes a `vector` column until you say `CREATE INDEX ... USING hnsw` (or `ivfflat`) yourself. Every query in notebook 01 was an exact sequential scan, and at 36 rows that is the optimal plan. Two more things that do *not* happen by themselves:

- a plain `CREATE INDEX` (B-tree) cannot index vectors; vector indexes require an operator class (`vector_cosine_ops`, ...),
- the planner will ignore even a perfect HNSW index while the table is small, because a seq scan is genuinely cheaper.

Plan for this notebook: scale to 50k rows, watch exact search become the bottleneck, build HNSW, measure the speedup, then face the two classic gotchas: **approximate recall** and **filters shrinking your results**.

In [1]:
import os, re, time
import numpy as np
import pandas as pd
import psycopg
from pgvector.psycopg import register_vector

conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True)
register_vector(conn)
cur = conn.cursor()

def explain(sql, params=None):
    cur.execute("EXPLAIN (ANALYZE) " + sql, params)
    text = "\n".join(r[0] for r in cur.fetchall())
    return re.sub(r"\[[^\[\]]{60,}\]", "[<vector>]", text)   # keep plans readable

def best_ms(sql, params=None, reps=5):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); cur.execute(sql, params); cur.fetchall()
        ts.append((time.perf_counter() - t0) * 1e3)
    return min(ts)

D, N, K = 384, 50_000, 10
rng = np.random.default_rng(42)

## 1. 50,000 rows, and why exact search dies

Deck 06's scale story: 100M x 1024d fp32 is 400 GB; ours is a modest 50k x 384d, about 82 MB. Instead of uniform noise we generate **clustered** vectors (100 Gaussian clusters of 500 points, then normalized). That is what real embedding spaces look like: clustering them was literally last week's homework.

In [2]:
centers = rng.normal(size=(100, D)); centers /= np.linalg.norm(centers, axis=1, keepdims=True)
vecs = np.repeat(centers, 500, axis=0) + rng.normal(scale=0.05, size=(N, D))
vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)
cats = rng.choice(["embeddings", "queues", "apis"], size=N, p=[0.33, 0.34, 0.33])
cats[:1000] = "databases"          # a rare category: 2%
rng.shuffle(cats)

cur.execute("DROP TABLE IF EXISTS documents_large")
cur.execute("""
    CREATE TABLE documents_large (
        id SERIAL PRIMARY KEY, title TEXT NOT NULL,
        category TEXT NOT NULL, embedding vector(384) NOT NULL)
""")
t0 = time.perf_counter()
with cur.copy("COPY documents_large (title, category, embedding) FROM STDIN") as cp:
    for i in range(N):
        cp.write_row((f"synthetic doc {i}", cats[i], vecs[i]))
print(f"COPY {N:,} rows: {time.perf_counter()-t0:.1f}s")
cur.execute("ANALYZE documents_large")

COPY 50,000 rows: 10.7s


<psycopg.Cursor [COMMAND_OK] [IDLE] (host=db user=demo database=vectordb) at 0xffff59637050>

`COPY` streams rows in one pass; `executemany` would work too but slower. Ground truth first, **before** any index exists: the exact top-10 for 20 query vectors, saved for the recall measurement later.

In [3]:
TOP = "SELECT id FROM documents_large ORDER BY embedding <=> %s LIMIT %s"

print(explain(TOP, (vecs[7], K)))
seq_ms = best_ms(TOP, (vecs[7], K))
print(f"\nexact latency (best of 5): {seq_ms:.1f} ms")

qids = rng.choice(N, 20, replace=False)
truth = {}
for qid in qids:
    cur.execute(TOP, (vecs[qid], K))
    truth[qid] = [r[0] for r in cur.fetchall()]
print("ground truth saved for", len(truth), "queries")

Limit  (cost=11753.48..11753.51 rows=10 width=12) (actual time=20.441..20.442 rows=10 loops=1)
  ->  Sort  (cost=11753.48..11878.48 rows=50000 width=12) (actual time=20.439..20.440 rows=10 loops=1)
        Sort Key: ((embedding <=> '[<vector>]'::vector))
        Sort Method: top-N heapsort  Memory: 25kB
        ->  Seq Scan on documents_large  (cost=0.00..10673.00 rows=50000 width=12) (actual time=0.020..17.557 rows=50000 loops=1)
Planning Time: 0.077 ms
Execution Time: 20.458 ms



exact latency (best of 5): 17.1 ms


ground truth saved for 20 queries


A parallel seq scan sorting all 50k distances: exact, correct, and linear in N. Fine at 50k, fatal at 100M.

## 2. Build the HNSW index

HNSW is a navigable small-world graph: a hierarchy of layers where a greedy walk descends from a sparse top layer to a dense bottom one, visiting only a tiny fraction of nodes per query. Two build knobs, defaults are sane: `m` (links per node, 16) and `ef_construction` (candidate list while building, 64).

In [4]:
t0 = time.perf_counter()
cur.execute("CREATE INDEX documents_large_hnsw ON documents_large"
            " USING hnsw (embedding vector_cosine_ops)")
build_s = time.perf_counter() - t0
cur.execute("ANALYZE documents_large")
print(f"index build: {build_s:.1f}s (one-time cost)")

print(explain(TOP, (vecs[7], K)))
hnsw_ms = best_ms(TOP, (vecs[7], K))
print(f"\nHNSW latency (best of 5): {hnsw_ms:.2f} ms  ->  {seq_ms/hnsw_ms:.0f}x faster than exact")

index build: 11.7s (one-time cost)
Limit  (cost=638.72..656.83 rows=10 width=12) (actual time=0.343..0.379 rows=10 loops=1)
  ->  Index Scan using documents_large_hnsw on documents_large  (cost=638.72..91196.00 rows=50000 width=12) (actual time=0.343..0.378 rows=10 loops=1)
        Order By: (embedding <=> '[<vector>]'::vector)
Planning Time: 0.057 ms
Execution Time: 0.389 ms

HNSW latency (best of 5): 0.20 ms  ->  87x faster than exact


The plan flipped to `Index Scan` and the sort is gone. Two prices, both visible:

In [5]:
sizes = {}
for rel, label in [("documents_large", "table (50k x 384d fp32)"),
                   ("documents_large_hnsw", "HNSW index")]:
    cur.execute("SELECT pg_relation_size(%s)", (rel,))
    sizes[label] = cur.fetchone()[0] / 1e6
    print(f"{label:26s} {sizes[label]:7.1f} MB")

table (50k x 384d fp32)       82.3 MB
HNSW index                   102.4 MB


The index is **larger than the table it serves**. That is normal for HNSW (each node stores `m` neighbor links) and worth saying out loud in class: an ANN index trades disk and build time for query latency.

## 3. Approximate means recall is a dial

`hnsw.ef_search` (default 40) is how many candidates the graph walk keeps per query. Small values are fast and lossy; large values approach exact search. We measure recall@10 against the ground truth saved in section 1:

In [6]:
def recall_at(ef):
    cur.execute(f"SET hnsw.ef_search = {int(ef)}")
    hits = []
    for qid in qids:
        cur.execute(TOP, (vecs[qid], K))
        got = [r[0] for r in cur.fetchall()]
        hits.append(len(set(got) & set(truth[qid])) / K)
    return float(np.mean(hits))

recalls = {}
for ef in [1, 8, 40, 200]:
    recalls[ef] = recall_at(ef)
    print(f"ef_search = {ef:4d}   recall@10 = {recalls[ef]:.2f}")
cur.execute("SET hnsw.ef_search = 40")   # back to default

ef_search =    1   recall@10 = 0.09
ef_search =    8   recall@10 = 0.73
ef_search =   40   recall@10 = 0.94
ef_search =  200   recall@10 = 0.98


<psycopg.Cursor [COMMAND_OK] [IDLE] (host=db user=demo database=vectordb) at 0xffff59637050>

At the default we are near-perfect on this data, and cranking `ef_search` down shows the approximation honestly. The knob is per-session, so different clients can trade differently.

## 4. One index per operator

We built the index for cosine (`vector_cosine_ops`). An L2 query cannot use it:

In [7]:
print(explain("SELECT id FROM documents_large ORDER BY embedding <-> %s LIMIT 5", (vecs[7],)))

Limit  (cost=11503.48..11503.49 rows=5 width=12) (actual time=20.128..20.129 rows=5 loops=1)
  ->  Sort  (cost=11503.48..11628.48 rows=50000 width=12) (actual time=20.127..20.128 rows=5 loops=1)
        Sort Key: ((embedding <-> '[<vector>]'::vector))
        Sort Method: top-N heapsort  Memory: 25kB
        ->  Seq Scan on documents_large  (cost=0.00..10673.00 rows=50000 width=12) (actual time=0.012..17.246 rows=50000 loops=1)
Planning Time: 0.030 ms
Execution Time: 20.159 ms


Back to a seq scan. If you need `<->` at scale, build a second index with `vector_l2_ops`; each operator class is its own index.

## 5. Filters + ANN: the shrinking-results gotcha

Here is the trap that makes "vector search with metadata filtering" its own topic. With the default `hnsw.iterative_scan = off`, the index walk produces ~`ef_search` candidates, Postgres **post-filters** them by `category`, and only then applies `LIMIT k`. Ask for the 5 nearest `databases` docs (a rare 2% category):

In [8]:
FILT = ("SELECT id, title FROM documents_large WHERE category = 'databases'"
        " ORDER BY embedding <=> %s LIMIT 5")

cur.execute(FILT, (vecs[7],))
shrink_rows = cur.fetchall()
print(f"asked for 5, got {len(shrink_rows)}\n")
print(explain(FILT, (vecs[7],)))

asked for 5, got 1

Limit  (cost=638.72..1092.88 rows=5 width=31) (actual time=0.182..0.189 rows=1 loops=1)
  ->  Index Scan using documents_large_hnsw on documents_large  (cost=638.72..91198.49 rows=997 width=31) (actual time=0.182..0.189 rows=1 loops=1)
        Order By: (embedding <=> '[<vector>]'::vector)
        Filter: (category = 'databases'::text)
        Rows Removed by Filter: 39
Planning Time: 0.018 ms
Execution Time: 0.194 ms


The plan even shows the evidence: `Rows Removed by Filter: ~40`. Almost nothing in the top-40 candidates was a `databases` row, so the LIMIT starved. A common category does not trigger it:

In [9]:
cur.execute(FILT.replace("'databases'", "'embeddings'"), (vecs[7],))
print(f"common category: asked for 5, got {len(cur.fetchall())}")

common category: asked for 5, got 5


Two honest fixes:

1. `SET hnsw.iterative_scan = strict_order` (pgvector 0.8+): if the first walk yields too few survivors, keep walking. Exact ordering, a bit more latency.
2. Do nothing: for a filter this selective the planner may rightly choose a seq scan over the filtered rows anyway; exact and correct.

In [10]:
cur.execute("SET hnsw.iterative_scan = strict_order")
cur.execute(FILT, (vecs[7],))
iter_rows = cur.fetchall()
print(f"strict_order: asked for 5, got {len(iter_rows)}\n")
print(explain(FILT, (vecs[7],)))
cur.execute("SET hnsw.iterative_scan = off")

strict_order: asked for 5, got 5

Limit  (cost=638.72..1092.88 rows=5 width=31) (actual time=0.191..0.460 rows=5 loops=1)
  ->  Index Scan using documents_large_hnsw on documents_large  (cost=638.72..91198.49 rows=997 width=31) (actual time=0.190..0.459 rows=5 loops=1)
        Order By: (embedding <=> '[<vector>]'::vector)
        Filter: (category = 'databases'::text)
        Rows Removed by Filter: 223
Planning Time: 0.017 ms
Execution Time: 0.464 ms


<psycopg.Cursor [COMMAND_OK] [IDLE] (host=db user=demo database=vectordb) at 0xffff59637050>

## 6. A partial index when one filter dominates

If one predicate covers most of your traffic, index only that slice. The planner picks it up automatically, no hints:

In [11]:
t0 = time.perf_counter()
cur.execute("CREATE INDEX documents_large_dbs ON documents_large"
            " USING hnsw (embedding vector_cosine_ops) WHERE category = 'databases'")
cur.execute("ANALYZE documents_large")
print(f"partial index build: {time.perf_counter()-t0:.1f}s")

cur.execute(FILT, (vecs[7],))
print(f"asked for 5, got {len(cur.fetchall())}\n")
print(explain(FILT, (vecs[7],)))

cur.execute("SELECT pg_relation_size('documents_large_dbs')")
partial_mb = cur.fetchone()[0] / 1e6
print(f"\npartial index size: {partial_mb:.1f} MB vs {sizes['HNSW index']:.1f} MB full")

partial index build: 0.2s
asked for 5, got 5

Limit  (cost=410.04..432.16 rows=5 width=31) (actual time=0.197..0.199 rows=5 loops=1)
  ->  Index Scan using documents_large_dbs on documents_large  (cost=410.04..4844.04 rows=1002 width=31) (actual time=0.197..0.199 rows=5 loops=1)
        Order By: (embedding <=> '[<vector>]'::vector)
Planning Time: 0.018 ms
Execution Time: 0.203 ms

partial index size: 2.1 MB vs 102.4 MB full


## 7. Bonus: half precision, half the index

Deck 06's fp16 slide, in index form. `halfvec` stores 2 bytes per dimension; an **expression index** over the cast gives you a half-size HNSW without touching the table:

In [12]:
t0 = time.perf_counter()
cur.execute("CREATE INDEX documents_large_hb ON documents_large"
            " USING hnsw ((embedding::halfvec(384)) halfvec_cosine_ops)")
print(f"halfvec build: {time.perf_counter()-t0:.1f}s")
cur.execute("SELECT pg_relation_size('documents_large_hb')")
hb_mb = cur.fetchone()[0] / 1e6
print(f"halfvec index: {hb_mb:.1f} MB vs fp32 {sizes['HNSW index']:.1f} MB\n")

HQ = ("SELECT id FROM documents_large"
      " ORDER BY embedding::halfvec(384) <=> %s::halfvec(384) LIMIT 5")
print(explain(HQ, (vecs[7],)))

halfvec build: 6.2s
halfvec index: 58.5 MB vs fp32 102.4 MB

Limit  (cost=367.05..374.00 rows=5 width=12) (actual time=0.378..0.384 rows=5 loops=1)
  ->  Index Scan using documents_large_hb on documents_large  (cost=367.05..69893.00 rows=50000 width=12) (actual time=0.377..0.383 rows=5 loops=1)
        Order By: ((embedding)::halfvec(384) <=> '[<vector>]'::halfvec(384))
Planning Time: 0.087 ms
Execution Time: 0.411 ms


## Summary (numbers from this run)

`documents_large` is left in place so you can keep poking at it; `docker compose down -v` wipes everything.

In [13]:
summary = pd.DataFrame([
    ("exact seq scan, 50k rows", f"{seq_ms:.1f} ms"),
    ("HNSW query", f"{hnsw_ms:.2f} ms ({seq_ms/hnsw_ms:.0f}x)"),
    ("HNSW build (one-time)", f"{build_s:.0f} s"),
    ("recall@10  ef=1 / 8 / 40 / 200", " / ".join(f"{recalls[e]:.2f}" for e in [1, 8, 40, 200])),
    ("rare filter, iterative off", f"asked 5, got {len(shrink_rows)}"),
    ("rare filter, strict_order", f"asked 5, got {len(iter_rows)}"),
    ("fp32 / halfvec / partial index",
     f"{sizes['HNSW index']:.0f} / {hb_mb:.0f} / {partial_mb:.0f} MB"),
], columns=["what", "measured"])
summary

,what,measured
0,"exact seq scan, 50k rows",17.1 ms
1,HNSW query,0.20 ms (87x)
2,HNSW build (one-time),12 s
3,recall@10 ef=1 / 8 / 40 / 200,0.09 / 0.73 / 0.94 / 0.98
4,"rare filter, iterative off","asked 5, got 1"
5,"rare filter, strict_order","asked 5, got 5"
6,fp32 / halfvec / partial index,102 / 59 / 2 MB
